In [12]:
import pandas as pd
import sqlite3

# =====================================================================
# STEP 1: LOAD & DYNAMICALLY CLEAN COLUMNS
# =====================================================================
print("--- STEP 1: Loading Dataset ---")
url = "https://raw.githubusercontent.com/PacktPublishing/Learning-Tableau-10/master/Chapter%2001/Superstore.csv"

# Load the raw data safely
df = pd.read_csv(url, encoding='unicode_escape')

# Lowercase everything and replace spaces/hyphens with underscores
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_').str.replace('–', '_')

# Build a dynamic mapper to find the columns our SQL script relies on
mapping = {}
for col in df.columns:
    if 'sub' in col:
        mapping[col] = 'sub_category'
    elif 'cust' in col and 'name' in col:
        mapping[col] = 'customer_name'
    elif 'cust' in col and 'id' in col:
        mapping[col] = 'customer_id'
    elif 'prod' in col and 'name' in col:
        mapping[col] = 'product_name'
    elif 'ord' in col and 'id' in col:
        mapping[col] = 'order_id'
    elif 'ord' in col and 'date' in col:
        mapping[col] = 'order_date'

# Apply the clean names to our dataframe
df = df.rename(columns=mapping)

# Quick safety fallback: ensure every required column physically exists
required_cols = ['order_id', 'order_date', 'customer_name', 'region', 'category', 'sub_category', 'product_name', 'sales', 'profit']
for col in required_cols:
    if col not in df.columns:
        df[col] = "N/A"

# Ensure calculation data types are clean numbers
df['sales'] = pd.to_numeric(df['sales'], errors='coerce').fillna(0)
df['profit'] = pd.to_numeric(df['profit'], errors='coerce').fillna(0)

# Connect to a fresh, in-memory SQLite Database
conn = sqlite3.connect(':memory:')
df.to_sql('superstore', conn, index=False, if_exists='replace')
print(f"Success! Dynamic mapper resolved columns. Database active with {len(df)} rows.\n")


# Helper function to execute queries and print them nicely
def run_query(title, query):
    print(f"\n--- {title} ---")
    try:
        result = pd.read_sql_query(query, conn)
        print(result.head(5).to_string(index=False)) # Display cleanly without index numbers
        return result
    except Exception as e:
        print(f"Error running query: {e}")

# =====================================================================
# STEP 2: EXPLORE TABLE
# =====================================================================
query_2 = "SELECT order_id, order_date, customer_name, category, sub_category, sales FROM superstore LIMIT 5;"
run_query("STEP 2: Sample Data Preview", query_2)


# =====================================================================
# STEP 3: APPLY WHERE FILTERS
# =====================================================================
query_3 = """
SELECT order_id, order_date, customer_name, region, category, sales
FROM superstore
WHERE region = 'West'
  AND category = 'Technology'
  AND sales >= 500;
"""
run_query("STEP 3: Filtered Data (West Region, Tech, Sales >= 500)", query_3)


# =====================================================================
# STEP 4: USE GROUP BY FOR AGGREGATIONS
# =====================================================================
query_4 = """
SELECT
    category,
    sub_category,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) AS total_revenue,
    AVG(sales) AS avg_order_value,
    SUM(profit) AS total_profit
FROM superstore
GROUP BY category, sub_category
ORDER BY total_revenue DESC;
"""
run_query("STEP 4: Category Performance Aggregations", query_4)


# =====================================================================
# STEP 5: SORT AND LIMIT RESULTS
# =====================================================================
query_5 = """
SELECT product_name, category, SUM(profit) AS total_product_profit
FROM superstore
GROUP BY product_name
ORDER BY total_product_profit DESC
LIMIT 5;
"""
run_query("STEP 5: Top 5 Most Profitable Products", query_5)


# =====================================================================
# STEP 6: SOLVE BUSINESS USE CASES
# =====================================================================
query_6a = """
SELECT
    SUBSTR(order_date, 1, 7) AS year_month,
    SUM(sales) AS monthly_sales,
    SUM(profit) AS monthly_profit
FROM superstore
GROUP BY year_month
ORDER BY year_month
LIMIT 5;
"""
run_query("STEP 6A: Monthly Trends (First 5 Months)", query_6a)

query_6b = """
SELECT customer_id, customer_name, SUM(sales) AS lifetime_spend
FROM superstore
GROUP BY customer_id, customer_name
ORDER BY lifetime_spend DESC
LIMIT 5;
"""
run_query("STEP 6B: Top 5 Highest-Value Customers", query_6b)


# =====================================================================
# STEP 7: VALIDATE RESULTS & DATA QUALITY
# =====================================================================
query_7 = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS missing_order_ids,
    SUM(CASE WHEN sales IS NULL OR sales <= 0 THEN 1 ELSE 0 END) AS invalid_sales_records
FROM superstore;
"""
run_query("STEP 7: Data Quality & Row Count Validation", query_7)

--- STEP 1: Loading Dataset ---
Success! Dynamic mapper resolved columns. Database active with 9426 rows.


--- STEP 2: Sample Data Preview ---
 order_id order_date      customer_name                       category sub_category  sales
    89128  11/8/2016        Mark Bailey                          Paper          N/A     53
    89128 11/11/2014        Mark Bailey                          Paper          N/A     76
    88745 11/23/2015    Daniel Richmond            Pens & Art Supplies          N/A     16
    86789  1/13/2016 Kristine Singleton Binders and Binder Accessories          N/A     65
    86789  1/13/2016 Kristine Singleton                   Rubber Bands          N/A     19

--- STEP 3: Filtered Data (West Region, Tech, Sales >= 500) ---
Empty DataFrame
Columns: [order_id, order_date, customer_name, region, category, sales]
Index: []

--- STEP 4: Category Performance Aggregations ---
                    category sub_category  total_orders  total_revenue  avg_order_value  total_p

,total_rows,missing_order_ids,invalid_sales_records
0,9426,0,0
